# 실습 1 신용카드 이상 탐지

데이터 준비
실습에 사용될 데이터는 Kaggle의 Credit Card Fraud Detection 데이터셋입니다. 이 데이터셋은 거래의 시간, 금액과 함께 28개의 PCA 변환된 특성들을 포함하고 있습니다. 'Class' 레이블은 사기 거래를 나타내는 1과 정상 거래를 나타내는 0으로 구분됩니다.

데이터를 불러오고, 전처리하는 기본적인 코드는 아래와 같습니다:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# 데이터를 불러옵니다.
csv_data = pd.read_csv('creditcard.csv')
print(csv_data)

            Time         V1         V2        V3        V4        V5  \
0            0.0  -1.359807  -0.072781  2.536347  1.378155 -0.338321   
1            0.0   1.191857   0.266151  0.166480  0.448154  0.060018   
2            1.0  -1.358354  -1.340163  1.773209  0.379780 -0.503198   
3            1.0  -0.966272  -0.185226  1.792993 -0.863291 -0.010309   
4            2.0  -1.158233   0.877737  1.548718  0.403034 -0.407193   
...          ...        ...        ...       ...       ...       ...   
284802  172786.0 -11.881118  10.071785 -9.834783 -2.066656 -5.364473   
284803  172787.0  -0.732789  -0.055080  2.035030 -0.738589  0.868229   
284804  172788.0   1.919565  -0.301254 -3.249640 -0.557828  2.630515   
284805  172788.0  -0.240440   0.530483  0.702510  0.689799 -0.377961   
284806  172792.0  -0.533413  -0.189733  0.703337 -0.506271 -0.012546   

              V6        V7        V8        V9  ...       V21       V22  \
0       0.462388  0.239599  0.098698  0.363787  ... -0.01830

In [4]:
# target 데이터(class)의 분포를 확인합니다.
print("Class Distribution:")
print(csv_data['Class'].value_counts())

# 위 모든 feature를 사용해서, class를 예측
data = csv_data.drop(['Time', 'Class'], axis = 1)
target = csv_data['Class']

Class Distribution:
Class
0    284315
1       492
Name: count, dtype: int64


In [5]:
# 데이터를 훈련 세트와 테스트 세트로 분할합니다.
from sklearn.model_selection import train_test_split
훈련용_data, 테스트용_data, 훈련용_target, 테스트용_target = train_test_split(
    data, target, test_size = 0.2, random_state = 40)

# 데이터 표준화 작업을 실시합니다,
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(훈련용_data)
ss.fit(테스트용_data)

표준화_훈련용_data = ss.transform(훈련용_data)
표준화_테스트용_data = ss.transform(테스트용_data)

In [6]:
# 로지스틱 회귀 모델을 생성하고 학습합니다.
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(표준화_훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
lr_pred = lr.predict(표준화_테스트용_data)
print(lr_pred)


# 정확도, 정밀도, F1 Score를 계산 및 출력합니다.
from sklearn.metrics import classification_report

print(classification_report(테스트용_target, lr_pred))


# AUC 점수를 계산합니다.
lr_auc_score = roc_auc_score(테스트용_target, lr.predict_proba(표준화_테스트용_data)[:, 1])
print("Logistic Regression AUC score:", lr_auc_score)

[0 0 0 ... 0 0 0]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56865
           1       0.89      0.65      0.75        97

    accuracy                           1.00     56962
   macro avg       0.94      0.82      0.87     56962
weighted avg       1.00      1.00      1.00     56962

Logistic Regression AUC score: 0.9790574710768224


## Decision Tree 로 직접해보기

In [7]:
# 결정 트리 모델을 생성하고 학습합니다.
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=10)

dt.fit(훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
print(dt.predict(테스트용_data))

# AUC 점수를 계산합니다.
dt_auc_score = roc_auc_score(테스트용_target, dt.predict_proba(테스트용_data)[:, 1])
print("Decision Tree AUC score:", dt_auc_score)

[0 0 0 ... 0 0 0]
Decision Tree AUC score: 0.9224430079923421


## Random Forest 로 해보기

In [8]:
# 랜덤 포레스트 모델을 생성하고 학습합니다.
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier(n_estimators = 10, n_jobs = -1, random_state = 40)

rfc.fit(훈련용_data, 훈련용_target)

# 학습된 모델로 테스트 데이터를 예측하고 평가합니다.
print(rfc.predict(테스트용_data))

# AUC 점수를 계산합니다.
rfc_auc_score = roc_auc_score(테스트용_target, rfc.predict_proba(테스트용_data)[:, 1])
print("Random Forest AUC score:", rfc_auc_score)

[0 0 0 ... 0 0 0]
Random Forest AUC score: 0.9481963521851808


## 퀴즈) SVM 사용해보기

# 가장 AUC점수가 높았던 모델 GridSearch로 튜닝하기

In [ ]:
# Feature를 10개로 줄이기
# Feature_Importance
# 상관관계를 구해서, 상관관계가 높은 항목들을 병합

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# 1. train.csv 데이터 로드
df_train = pd.read_csv('train.csv')

# 2. 마지막 열은 무조건 타겟(y), 나머지는 독립변수(X)로 분리 (이름 무관)
X_train = df_train.iloc[:, :-1]
y_train = df_train.iloc[:, -1]

# 3. [파생변수 조합] 이름 몰라도 위치로 자동 조립
base_col_name = X_train.columns[-1] 
t1, t2 = X_train.columns[0], X_train.columns[1]

X_train['New_Div_Feature'] = X_train[t1] / (X_train[base_col_name] + 1e-5)
X_train['New_Weighted_Feature'] = (X_train[t1] * 2.0) + X_train[t2]

# 4. 종합 연관성 순위 계산 및 상위 5개 '위치(인덱스 번호)' 추출
corr_rank = X_train.corrwith(y_train).abs().rank(ascending=False)
rf_final = RandomForestClassifier(n_estimators=10, class_weight='balanced', n_jobs=-1, random_state=40)
rf_final.fit(X_train.values, y_train.values) # .values로 이름 지우기
rf_rank = pd.Series(rf_final.feature_importances_, index=X_train.columns).rank(ascending=False)

# 컬럼의 '위치 번호(숫자)'를 리스트로 추출
top_5_indices = (corr_rank + rf_rank).sort_values().index[:5]
top_5_pos = [X_train.columns.get_loc(col) for col in top_5_indices]

# 5. 모델 학습 진행 (X_train_final 뒤에 .values를 붙여 컬럼명을 완전히 파괴합니다)
X_train_final = X_train[top_5_indices].values # ★ 핵심: 컬럼 이름 제거

final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_final)

best_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=40)
best_model.fit(X_train_scaled, y_train.values) # ★ 핵심: 타겟 이름도 제거

# 6. 규칙 파일 보관
joblib.dump(best_model, 'best_rf_model.pkl')
joblib.dump(final_scaler, 'final_scaler.pkl')
joblib.dump(top_5_pos, 'top_5_pos.pkl')

print("=" * 60)
print("▶ [성공] 훈련을 마쳤습니다.")
print(f"▶ 선정된 열 위치 번호: {top_5_pos}")
print("=" * 60)

▶ [성공] 훈련을 마쳤습니다.
▶ 선정된 열 위치 번호: [7, 11, 1, 6, 3]


In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report, f1_score

def predict_realtime_test_set(test_csv_path):
    """
    이름이 다르면 .values를 이용해 순수한 숫자 데이터만 추출,
    훈련 때 기억한 '위치 번호'로만 매핑하여 예측하는 무결점 함수
    """
    df_test = pd.read_csv(test_csv_path)
    
    # 1. 파일에서 규칙과 모델 로드
    loaded_pos = joblib.load('top_5_pos.pkl')
    loaded_scaler = joblib.load('final_scaler.pkl')
    loaded_model = joblib.load('best_rf_model.pkl')
    
    # 2. test.csv도 마지막 열(Class)을 제외하고 복제
    X_test = df_test.iloc[:, :-1]
    
    # 3. 훈련 데이터와 동일한 '위치'의 열들을 찾아 파생변수 강제 생성
    test_base_col = X_test.columns[-1]
    test_t1, test_t2 = X_test.columns[0], X_test.columns[1]
    
    X_test['New_Div_Feature'] = X_test[test_t1] / (X_test[test_base_col] + 1e-5)
    X_test['New_Weighted_Feature'] = (X_test[test_t1] * 2.0) + X_test[test_t2]
    
    # 4. 저장되었던 '위치 번호'를 기반으로 5개 열만 필터링
    final_cols = []
    for pos in loaded_pos:
        final_cols.append(X_test.columns[pos])
        
    # ★ 핵심: 필터링한 데이터 뒤에 .values를 붙여 이름 정보를 완벽히 지워줍니다.
    X_realtime = X_test[final_cols].values 
    
    # 5. 전처리 및 예측 (이제 모델이 이름 시비를 걸지 않습니다!)
    X_realtime_scaled = loaded_scaler.transform(X_realtime)
    return loaded_model.predict(X_realtime_scaled)

# --- [실시간 가동 및 채점] ---
test_file_name = 'test.csv'

print("▶ [가동] 이름 독립형 위치 기반 실시간 테스트를 시작합니다...")
final_predictions = predict_realtime_test_set(test_file_name)

print("\n" + "="*60)
print("★ [최종 성적표] 실시간 구동 모델 성능 채점 결과")
print("="*60)

df_actual = pd.read_csv(test_file_name)
y_actual = df_actual.iloc[:, -1].values # 정답 라벨도 순수 배열로 변경

print(classification_report(y_actual, final_predictions, digits=4))
macro_f1 = f1_score(y_actual, final_predictions, average='macro')
print("-"*60)
print(f"🎯 교수님 제출용 기말고사 최종 평가 점수 (Macro F1): {macro_f1:.4f}")
print("="*60)

▶ [가동] 이름 독립형 위치 기반 실시간 테스트를 시작합니다...

★ [최종 성적표] 실시간 구동 모델 성능 채점 결과
              precision    recall  f1-score   support

           0     0.9670    1.0000    0.9832        88
           1     1.0000    0.7500    0.8571        12

    accuracy                         0.9700       100
   macro avg     0.9835    0.8750    0.9202       100
weighted avg     0.9710    0.9700    0.9681       100

------------------------------------------------------------
🎯 교수님 제출용 기말고사 최종 평가 점수 (Macro F1): 0.9202
